In [4]:
pip install intake-esm xarray gcsfs netCDF4 zarr


   ---------------------------------------- 0.0/38.0 MB ? eta -:--:--
    --------------------------------------- 0.5/38.0 MB 3.5 MB/s eta 0:00:11
   - -------------------------------------- 1.6/38.0 MB 4.6 MB/s eta 0:00:08
   -- ------------------------------------- 2.4/38.0 MB 3.9 MB/s eta 0:00:10
   --- ------------------------------------ 3.1/38.0 MB 4.1 MB/s eta 0:00:09
   ---- ----------------------------------- 3.9/38.0 MB 4.1 MB/s eta 0:00:09
   ----- ---------------------------------- 5.0/38.0 MB 4.1 MB/s eta 0:00:09
   ------ --------------------------------- 6.0/38.0 MB 4.2 MB/s eta 0:00:08
   ------- -------------------------------- 7.1/38.0 MB 4.3 MB/s eta 0:00:08
   -------- ------------------------------- 8.4/38.0 MB 4.5 MB/s eta 0:00:07
   --------- ------------------------------ 9.4/38.0 MB 4.6 MB/s eta 0:00:07
   ---------- ----------------------------- 10.2/38.0 MB 4.5 MB/s eta 0:00:07
   ----------- ---------------------------- 10.7/38.0 MB 4.4 MB/s eta 0:00:07
   

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
s3fs 2025.3.2 requires fsspec==2025.3.2.*, but you have fsspec 2026.3.0 which is incompatible.


In [15]:
import pandas as pd
import numpy as np
import xarray as xr
import gcsfs

fs = gcsfs.GCSFileSystem(token="anon")

df_cat = pd.read_csv(
    "https://cmip6.storage.googleapis.com/pangeo-cmip6.csv"
)

cities = {
    "Miami FL": (25.77, -80.19),
    "New Orleans LA": (29.95, -90.07),
    "Norfolk VA": (36.85, -76.29),
    "New York NY": (40.71, -74.01),
    "Atlantic City NJ": (39.36, -74.43),
    "Charleston SC": (32.78, -79.93),
    "Tampa FL": (27.95, -82.46),
    "Galveston TX": (29.30, -94.79),
    "Honolulu HI": (21.31, -157.86),
    "Seattle WA": (47.61, -122.33),
    "Boston MA": (42.36, -71.06),
    "San Francisco CA": (37.77, -122.42),
}

def get_lat_lon_names(ds):
    lat_name = next(name for name in ["lat", "latitude", "nav_lat"] if name in ds)
    lon_name = next(name for name in ["lon", "longitude", "nav_lon"] if name in ds)
    return lat_name, lon_name


def nearest_valid_ocean_timeseries(ds, data_array, city_lat, city_lon):
    lat_name, lon_name = get_lat_lon_names(ds)

    lats = ds[lat_name]
    lons = ds[lon_name]

    if float(lons.max()) > 180 and city_lon < 0:
        city_lon += 360

    # Use first time slice to find valid ocean cells
    first_slice = data_array.isel(time=0)
    valid_mask = np.isfinite(first_slice)

    dist = np.sqrt((lats - city_lat) ** 2 + (lons - city_lon) ** 2)

    # Only allow valid ocean cells
    dist = dist.where(valid_mask)

    y_dim, x_dim = lats.dims

    flat_index = int(np.nanargmin(dist.values))
    y_idx, x_idx = np.unravel_index(flat_index, dist.shape)

    return data_array.isel({y_dim: y_idx, x_dim: x_idx})


hist_subset = (
    df_cat.query(
        "variable_id == 'zos' & experiment_id == 'historical' & table_id == 'Omon'"
    )
    .drop_duplicates("source_id")
)

records = []

for _, row in hist_subset.head(10).iterrows():
    try:
        print("Loading", row.source_id)

        ds = xr.open_zarr(
            fs.get_mapper(row.zstore),
            consolidated=True
        )

        annual = (
            ds["zos"]
            .sel(time=slice("1850", "2014"))
            .resample(time="1YE")
            .mean()
        )

        for city, (lat, lon) in cities.items():
            ts = nearest_valid_ocean_timeseries(ds, annual, lat, lon).values
            print(city, np.isnan(ts).sum(), "missing out of", len(ts))
            print(ts[:5])
            for yr_idx, yr in enumerate(range(1850, 2015)):
                records.append({
                    "city": city,
                    "year": yr,
                    "segment": "historical_cmip6",
                    "scenario": "historical",
                    "model": row.source_id,
                    "dynamic_slr_cm": round(float(ts[yr_idx]), 2),
                })

    except Exception as e:
        print(f"Skipped {row.source_id}: {e}")
        continue

historical_df = pd.DataFrame(records)
historical_df.to_csv("data/historical_city_sea_level.csv", index=False)

historical_df.head()

Loading GFDL-ESM4
Miami FL 0 missing out of 165
[-0.23162723 -0.24209867 -0.2355633  -0.22925788 -0.2522898 ]
New Orleans LA 0 missing out of 165
[-0.16535619 -0.17288889 -0.1806159  -0.14503533 -0.16027786]
Norfolk VA 0 missing out of 165
[-0.668608  -0.7008651 -0.6744428 -0.6688878 -0.6592191]
New York NY 0 missing out of 165
[-0.68819046 -0.7147339  -0.6893373  -0.7059328  -0.6810608 ]
Atlantic City NJ 0 missing out of 165
[-0.6948169  -0.7256658  -0.69465995 -0.70711875 -0.68684393]
Charleston SC 0 missing out of 165
[-0.5143228  -0.5185407  -0.5319101  -0.4799812  -0.49064216]
Tampa FL 0 missing out of 165
[-0.22796404 -0.24864316 -0.23824877 -0.21719174 -0.24413775]
Galveston TX 0 missing out of 165
[-0.21370476 -0.21721707 -0.22688083 -0.1807652  -0.2094237 ]
Honolulu HI 0 missing out of 165
[0.5485115  0.5040565  0.4836751  0.50900817 0.55059266]
Seattle WA 0 missing out of 165
[0.24902199 0.24460691 0.27238184 0.25976714 0.27341172]
Boston MA 0 missing out of 165
[-0.63214433 

,city,year,segment,scenario,model,dynamic_slr_cm
0,Miami FL,1850,historical_cmip6,historical,GFDL-ESM4,-0.23
1,Miami FL,1851,historical_cmip6,historical,GFDL-ESM4,-0.24
2,Miami FL,1852,historical_cmip6,historical,GFDL-ESM4,-0.24
3,Miami FL,1853,historical_cmip6,historical,GFDL-ESM4,-0.23
4,Miami FL,1854,historical_cmip6,historical,GFDL-ESM4,-0.25


In [16]:
scenarios = ['ssp126', 'ssp245', 'ssp370', 'ssp585']

future_records = []

for scenario in scenarios:

    print(f"\n=== Processing {scenario} ===")

    future_subset = (
        df_cat.query(
            f"variable_id == 'zos' & "
            f"experiment_id == '{scenario}' & "
            "table_id == 'Omon'"
        )
        .drop_duplicates("source_id")
    )

    for _, row in future_subset.head(10).iterrows():

        try:
            print(f"Loading {row.source_id}")

            ds = xr.open_zarr(
                fs.get_mapper(row.zstore),
                consolidated=True
            )

            annual = (
                ds["zos"]
                .sel(time=slice("2015", "2100"))
                .resample(time="1YE")
                .mean()
            )

            for city, (lat, lon) in cities.items():
                print(city, np.isnan(ts).sum(), "missing out of", len(ts))
                print(ts[:5])
                ts = nearest_valid_ocean_timeseries(
                    ds,
                    annual,
                    lat,
                    lon
                ).values

                for yr_idx, yr in enumerate(range(2015, 2101)):

                    future_records.append({
                        "city": city,
                        "year": yr,
                        "segment": "future_cmip6",
                        "scenario": scenario,
                        "model": row.source_id,
                        "dynamic_slr_cm": round(
                            float(ts[yr_idx]),
                            2
                        ),
                    })

        except Exception as e:
            print(f"Skipped {row.source_id}: {e}")
            continue


=== Processing ssp126 ===
Loading GFDL-ESM4
Miami FL 0 missing out of 165
[0.10905588 0.1240626  0.1804313  0.17990963 0.19821753]
New Orleans LA 0 missing out of 86
[-0.23893194 -0.21872927 -0.2089595  -0.22369814 -0.22949362]
Norfolk VA 0 missing out of 86
[-0.16283903 -0.13245998 -0.07772598 -0.11296517 -0.15784183]
New York NY 0 missing out of 86
[-0.6486004  -0.57721037 -0.5589343  -0.6509866  -0.6381387 ]
Atlantic City NJ 0 missing out of 86
[-0.6695662  -0.6231619  -0.6106015  -0.70456314 -0.69400406]
Charleston SC 0 missing out of 86
[-0.67473245 -0.6256977  -0.6101819  -0.6961689  -0.6845722 ]
Tampa FL 0 missing out of 86
[-0.49753186 -0.42634282 -0.4044524  -0.47487524 -0.4863131 ]
Galveston TX 0 missing out of 86
[-0.2216777  -0.21378358 -0.18496643 -0.2082743  -0.22706683]
Honolulu HI 0 missing out of 86
[-0.1990226  -0.17103706 -0.12401977 -0.14255069 -0.1950729 ]
Seattle WA 0 missing out of 86
[0.5310115  0.53764844 0.5278643  0.50370944 0.49339202]
Boston MA 0 missing o

In [17]:
_future_df = pd.DataFrame(future_records)
_future_df.to_csv("data/future_city_sea_level.csv", index=False)

_future_df.head()

,city,year,segment,scenario,model,dynamic_slr_cm
0,Miami FL,2015,future_cmip6,ssp126,GFDL-ESM4,-0.24
1,Miami FL,2016,future_cmip6,ssp126,GFDL-ESM4,-0.22
2,Miami FL,2017,future_cmip6,ssp126,GFDL-ESM4,-0.21
3,Miami FL,2018,future_cmip6,ssp126,GFDL-ESM4,-0.22
4,Miami FL,2019,future_cmip6,ssp126,GFDL-ESM4,-0.23


In [ ]:
combined_df = pd.concat([historical_df, _future_df], ignore_index=True)

,city,year,segment,scenario,model,dynamic_slr_cm
0,Miami FL,1850,historical_cmip6,historical,GFDL-ESM4,-0.23
1,Miami FL,1851,historical_cmip6,historical,GFDL-ESM4,-0.24
2,Miami FL,1852,historical_cmip6,historical,GFDL-ESM4,-0.24
3,Miami FL,1853,historical_cmip6,historical,GFDL-ESM4,-0.23
4,Miami FL,1854,historical_cmip6,historical,GFDL-ESM4,-0.25


In [21]:
city_coords = {
    "Miami FL": (25.77, -80.19),
    "New Orleans LA": (29.95, -90.07),
    "Norfolk VA": (36.85, -76.29),
    "New York NY": (40.71, -74.01),
    "Atlantic City NJ": (39.36, -74.43),
    "Charleston SC": (32.78, -79.93),
    "Tampa FL": (27.95, -82.46),
    "Galveston TX": (29.30, -94.79),
    "Honolulu HI": (21.31, -157.86),
    "Seattle WA": (47.61, -122.33),
    "Boston MA": (42.36, -71.06),
    "San Francisco CA": (37.77, -122.42),
}
combined_df["lat"] = combined_df["city"].map(
    lambda x: city_coords[x][0]
)

combined_df["lon"] = combined_df["city"].map(
    lambda x: city_coords[x][1]
)
combined_df.rename(columns={"dynamic_slr_cm": "zos_raw"}, inplace=True)
combined_df.to_csv("data/combined_city_sea_level.csv", index=False)